# Notebook de entrenamiento de modelos

En este Notebook, se entrenan tres modelos de Machine Learning:

- Support Vector Machine
- Decision Tree
- K-Nearest Neighbours

Estos modelos se usan para predecir a qué especie pertenece un pinguino a partir de sus características físicas.

In [1]:
!pip install boto3 sqlalchemy pymysql mlflow minio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.8/93.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 9.9 MB/s eta 0:00:00:00:0100:01

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


### Importación de librerías

In [4]:
import pandas as pd
from sqlalchemy import create_engine
import pymysql
import pandas as pd
import pickle
from pathlib import Path
import json
import os
import boto3

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix

### Función para leer tabla de data lista para entrenamiento

In [5]:
def load_processed_table():

    user = "mlops_user"
    password = "mlops_pass"      # cambia si tu docker usa otro
    host = "mysql_db"
    port = 3306
    database = "mlops_db"

    engine = create_engine(
        f"mysql+pymysql://{user}:{password}@{host}:{port}/{database}"
    )

    df = pd.read_sql_table("penguins_processed", engine)

    return df

In [6]:
df_processed = load_processed_table()

In [7]:
df_processed.head()

,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g,island_Biscoe,island_Dream,island_Torgersen,sex_FEMALE,sex_MALE,sex_Unknown,species
0,39.10,18.7,181.0,3750.0,0.0,0.0,1.0,0.0,1.0,0.0,Adelie
1,39.50,17.4,186.0,3800.0,0.0,0.0,1.0,1.0,0.0,0.0,Adelie
2,40.30,18.0,195.0,3250.0,0.0,0.0,1.0,1.0,0.0,0.0,Adelie
3,44.45,17.3,197.0,4050.0,0.0,0.0,1.0,0.0,0.0,1.0,Adelie
4,36.70,19.3,193.0,3450.0,0.0,0.0,1.0,1.0,0.0,0.0,Adelie


### Creación de Bucket para MLFLOW en MinIO

In [8]:
from minio import Minio
from minio.error import S3Error

# Configuración del cliente MinIO
client = Minio(
    "minio:9000",              
    access_key="admin",            
    secret_key="supersecret",      
    secure=False                   
)

bucket_name = "mlflows3"

# Crear bucket si no existe
if not client.bucket_exists(bucket_name):
    client.make_bucket(bucket_name)
    print(f"✅ Bucket '{bucket_name}' creado en MinIO")
else:
    print(f"ℹ️ Bucket '{bucket_name}' ya existe")


✅ Bucket 'mlflows3' creado en MinIO


## Entrenamiento de modelos

A continuación se definen las funciones para entrenar los modelos de ML y subir el modelo que mejor desempeño tiene a MLflow..

In [9]:
# ======================== Decision Tree con MLflow ========================

import os
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient
from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import pandas as pd


def train_decision_tree_with_mlflow(df, experiment_name="penguins_decision_tree"):
    """
    Entrena múltiples modelos de Árbol de Decisión con variaciones de hiperparámetros,
    registra cada ejecución en MLflow y selecciona el mejor modelo para producción.
    
    Args:
        df (pd.DataFrame): DataFrame preprocesado con las características y la columna 'species'.
        experiment_name (str): Nombre del experimento en MLflow.
        
    Returns:
        best_model: Modelo entrenado con mejor desempeño.
    """
    # Separar X e y
    X = df.drop(columns="species")
    y = df["species"]

    # División train/test
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.3,
        random_state=42,
        stratify=y
    )

    # Definir espacio de hiperparámetros (mínimo 20 combinaciones)
    param_grid = {
        "max_depth": [3, 4, 5, 6, 7],
        "min_samples_split": [2, 5, 10],
        "criterion": ["gini", "entropy"]
    }
    grid = list(ParameterGrid(param_grid))  # todas las combinaciones

    # Configuración MLflow
    mlflow.set_tracking_uri("http://mlflow:5000")
    mlflow.set_experiment(experiment_name)

    best_acc = 0
    best_run_id = None
    best_model = None

    # Iterar sobre todas las combinaciones
    for params in grid:
        with mlflow.start_run(run_name=f"DecisionTree_{params}") as run:
            # Entrenar modelo
            model = DecisionTreeClassifier(
                **params,
                class_weight="balanced",
                random_state=42
            )
            model.fit(X_train, y_train)

            # Evaluar
            y_pred = model.predict(X_test)
            acc = accuracy_score(y_test, y_pred)
            report = classification_report(y_test, y_pred)
            matrix = confusion_matrix(y_test, y_pred)

            # Loggear en MLflow
            mlflow.log_params(params)
            mlflow.log_metric("accuracy", acc)
            mlflow.log_text(report, "classification_report.txt")
            mlflow.log_text(str(matrix), "confusion_matrix.txt")

            # Guardar modelo en MLflow
            mlflow.sklearn.log_model(
                sk_model=model,
                artifact_path="decision_tree"
            )

            # Actualizar mejor modelo
            if acc > best_acc:
                best_acc = acc
                best_run_id = run.info.run_id
                best_model = model

    # Al final, después de encontrar el mejor run:
    client = MlflowClient()
    model_name = "penguins_decision_tree_model"

    # Construir la URI del modelo loggeado en el run
    model_uri = f"runs:/{best_run_id}/decision_tree"

    # Registrar modelo en el Model Registry
    mv = mlflow.register_model(
        model_uri=model_uri,
        name=model_name
    )

    # Pasar a producción
    client.transition_model_version_stage(
        name=model_name,
        version=mv.version,
        stage="Production"
    )

    print(f"✅ Mejor modelo registrado en MLflow con accuracy={best_acc:.4f} y puesto en Production")

    return best_model


In [10]:
# ======================== SVM con MLflow ========================

import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient
from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import pandas as pd

def train_svm_with_mlflow(df, experiment_name="penguins_svm"):
    """
    Entrena múltiples modelos SVM con variaciones de hiperparámetros,
    registra cada ejecución en MLflow y selecciona el mejor modelo para producción.
    """
    # Separar X e y
    X = df.drop(columns="species")
    y = df["species"]

    # Escalado
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    X = pd.DataFrame(X_scaled, columns=X.columns, index=X.index)

    # División train/test
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.3,
        random_state=42,
        stratify=y
    )

    # Espacio de hiperparámetros
    param_grid = {
        "C": [0.1, 1, 10],
        "kernel": ["linear", "rbf", "poly"],
        "gamma": ["scale", "auto"]
    }
    grid = list(ParameterGrid(param_grid))

    mlflow.set_tracking_uri("http://mlflow:5000")
    mlflow.set_experiment(experiment_name)

    best_acc = 0
    best_run_id = None
    best_model = None

    for params in grid:
        with mlflow.start_run(run_name=f"SVM_{params}") as run:
            model = SVC(
                **params,
                class_weight="balanced",
                probability=True,
                random_state=42
            )
            model.fit(X_train, y_train)

            y_pred = model.predict(X_test)
            acc = accuracy_score(y_test, y_pred)
            report = classification_report(y_test, y_pred)
            matrix = confusion_matrix(y_test, y_pred)

            mlflow.log_params(params)
            mlflow.log_metric("accuracy", acc)
            mlflow.log_text(report, "classification_report.txt")
            mlflow.log_text(str(matrix), "confusion_matrix.txt")

            mlflow.sklearn.log_model(
                sk_model=model,
                artifact_path="svm"
            )

            if acc > best_acc:
                best_acc = acc
                best_run_id = run.info.run_id
                best_model = model

    client = MlflowClient()
    model_name = "penguins_svm_model"
    model_uri = f"runs:/{best_run_id}/svm"

    mv = mlflow.register_model(
        model_uri=model_uri,
        name=model_name
    )

    client.transition_model_version_stage(
        name=model_name,
        version=mv.version,
        stage="Production"
    )

    print(f"✅ Mejor modelo SVM registrado en MLflow con accuracy={best_acc:.4f} y puesto en Production")
    return best_model


In [11]:
# ======================== KNN con MLflow ========================

from sklearn.neighbors import KNeighborsClassifier

def train_knn_with_mlflow(df, experiment_name="penguins_knn"):
    """
    Entrena múltiples modelos KNN con variaciones de hiperparámetros,
    registra cada ejecución en MLflow y selecciona el mejor modelo para producción.
    """
    # Separar X e y
    X = df.drop(columns="species")
    y = df["species"]

    # Escalado
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    X = pd.DataFrame(X_scaled, columns=X.columns, index=X.index)

    # División train/test
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.3,
        random_state=42,
        stratify=y
    )

    # Espacio de hiperparámetros
    param_grid = {
        "n_neighbors": [3, 5, 7, 9],
        "weights": ["uniform", "distance"],
        "p": [1, 2]  # distancia Manhattan o Euclídea
    }
    grid = list(ParameterGrid(param_grid))

    mlflow.set_tracking_uri("http://mlflow:5000")
    mlflow.set_experiment(experiment_name)

    best_acc = 0
    best_run_id = None
    best_model = None

    for params in grid:
        with mlflow.start_run(run_name=f"KNN_{params}") as run:
            model = KNeighborsClassifier(**params)
            model.fit(X_train, y_train)

            y_pred = model.predict(X_test)
            acc = accuracy_score(y_test, y_pred)
            report = classification_report(y_test, y_pred)
            matrix = confusion_matrix(y_test, y_pred)

            mlflow.log_params(params)
            mlflow.log_metric("accuracy", acc)
            mlflow.log_text(report, "classification_report.txt")
            mlflow.log_text(str(matrix), "confusion_matrix.txt")

            mlflow.sklearn.log_model(
                sk_model=model,
                artifact_path="knn"
            )

            if acc > best_acc:
                best_acc = acc
                best_run_id = run.info.run_id
                best_model = model

    client = MlflowClient()
    model_name = "penguins_knn_model"
    model_uri = f"runs:/{best_run_id}/knn"

    mv = mlflow.register_model(
        model_uri=model_uri,
        name=model_name
    )

    client.transition_model_version_stage(
        name=model_name,
        version=mv.version,
        stage="Production"
    )

    print(f"✅ Mejor modelo KNN registrado en MLflow con accuracy={best_acc:.4f} y puesto en Production")
    return best_model


In [12]:
# Decision Tree

train_decision_tree_with_mlflow(df_processed, experiment_name="penguins_decision_tree_4")

2026/03/21 09:00:16 INFO mlflow.tracking.fluent: Experiment with name 'penguins_decision_tree_4' does not exist. Creating a new experiment.
2026/03/21 09:00:18 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/21 09:00:19 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run DecisionTree_{'criterion': 'gini', 'max_depth': 3, 'min_samples_split': 2} at: http://mlflow:5000/#/experiments/1/runs/0ceebc1022f3468a82c2870c89e98564
🧪 View experiment at: http://mlflow:5000/#/experiments/1


2026/03/21 09:00:23 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/21 09:00:23 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run DecisionTree_{'criterion': 'gini', 'max_depth': 3, 'min_samples_split': 5} at: http://mlflow:5000/#/experiments/1/runs/6fea0e3a3d714a6b82a3b714a296ac53
🧪 View experiment at: http://mlflow:5000/#/experiments/1


2026/03/21 09:00:27 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/21 09:00:27 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/03/21 09:00:31 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


...

2026/03/21 09:02:20 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/21 09:02:21 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run DecisionTree_{'criterion': 'entropy', 'max_depth': 7, 'min_samples_split': 10} at: http://mlflow:5000/#/experiments/1/runs/4bea452c81dc45fbb87068da7141cc36
🧪 View experiment at: http://mlflow:5000/#/experiments/1


Successfully registered model 'penguins_decision_tree_model'.
2026/03/21 09:02:25 WARNING mlflow.tracking._model_registry.fluent: Run with id dc212bc81c8f4c889f2968405667914a has no artifacts at artifact path 'decision_tree', registering model based on models:/m-7c797f99bd1f489db581498134afc9cb instead
2026/03/21 09:02:26 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: penguins_decision_tree_model, version 1
Created version '1' of model 'penguins_decision_tree_model'.
/tmp/ipykernel_16/2745799193.py:102: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


✅ Mejor modelo registrado en MLflow con accuracy=0.9808 y puesto en Production


,criterion,'gini'
,splitter,'best'
,max_depth,5
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,'balanced'


In [13]:
# SVM

train_svm_with_mlflow(df_processed, experiment_name="penguins_svm")

2026/03/21 09:03:01 INFO mlflow.tracking.fluent: Experiment with name 'penguins_svm' does not exist. Creating a new experiment.
2026/03/21 09:03:01 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/21 09:03:01 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run SVM_{'C': 0.1, 'gamma': 'scale', 'kernel': 'linear'} at: http://mlflow:5000/#/experiments/2/runs/c55dd39fcc134b239306f6cf6ee383da
🧪 View experiment at: http://mlflow:5000/#/experiments/2


2026/03/21 09:03:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/21 09:03:05 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run SVM_{'C': 0.1, 'gamma': 'scale', 'kernel': 'rbf'} at: http://mlflow:5000/#/experiments/2/runs/940a879a7d414570a5835b07ddf5f25b
🧪 View experiment at: http://mlflow:5000/#/experiments/2


2026/03/21 09:03:10 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/21 09:03:10 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


...

🏃 View run SVM_{'C': 10, 'gamma': 'auto', 'kernel': 'poly'} at: http://mlflow:5000/#/experiments/2/runs/c697060789a3448f8b88bf6e6a714345
🧪 View experiment at: http://mlflow:5000/#/experiments/2


Successfully registered model 'penguins_svm_model'.
2026/03/21 09:04:05 WARNING mlflow.tracking._model_registry.fluent: Run with id 113c5c23c03a42909396e13694a762f1 has no artifacts at artifact path 'svm', registering model based on models:/m-58518cf7daa54eae8e26b90ea28e3f0e instead
2026/03/21 09:04:05 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: penguins_svm_model, version 1
Created version '1' of model 'penguins_svm_model'.


✅ Mejor modelo SVM registrado en MLflow con accuracy=0.9904 y puesto en Production


/tmp/ipykernel_16/1853697412.py:88: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


,C,1
,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,True
,tol,0.001
,cache_size,200
,class_weight,'balanced'
,verbose,False


In [ ]:
# KNN

train_knn_with_mlflow(df_processed, experiment_name="penguins_knn")

2026/03/21 09:04:05 INFO mlflow.tracking.fluent: Experiment with name 'penguins_knn' does not exist. Creating a new experiment.
2026/03/21 09:04:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/21 09:04:06 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run KNN_{'n_neighbors': 3, 'p': 1, 'weights': 'uniform'} at: http://mlflow:5000/#/experiments/3/runs/050eb16568c34e50946367cf028a1539
🧪 View experiment at: http://mlflow:5000/#/experiments/3


2026/03/21 09:04:09 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/21 09:04:09 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run KNN_{'n_neighbors': 3, 'p': 1, 'weights': 'distance'} at: http://mlflow:5000/#/experiments/3/runs/e4db97d09a2b43caa004e177dfbc95fa
🧪 View experiment at: http://mlflow:5000/#/experiments/3


2026/03/21 09:04:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/21 09:04:13 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


...

2026/03/21 09:04:37 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/21 09:04:37 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run KNN_{'n_neighbors': 7, 'p': 1, 'weights': 'distance'} at: http://mlflow:5000/#/experiments/3/runs/6ca05dcf03e94ba38c48e2a76db9c54b
🧪 View experiment at: http://mlflow:5000/#/experiments/3


2026/03/21 09:04:40 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/21 09:04:41 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run KNN_{'n_neighbors': 7, 'p': 2, 'weights': 'uniform'} at: http://mlflow:5000/#/experiments/3/runs/def8badf4e524de89535bfc79c372c77
🧪 View experiment at: http://mlflow:5000/#/experiments/3


2026/03/21 09:04:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/21 09:04:44 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


In [ ]:
# Fin del cuadernillo